# Training Diabetes Prediction Model

Training `RandomForestRegressor` on `sklearn.datasets.load_diabetes(scaled=False)` with `StandardScaler` pipeline.
Logging experiment to MLflow and registering the model.

In [2]:
import mlflow
import mlflow.sklearn
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

## 1. Load Data

In [3]:
data = load_diabetes(scaled=False)
X, y = data.data, data.target
feature_names = data.feature_names

print(f"Features: {feature_names}")
print(f"Dataset shape: {X.shape}")
print(f"Target range: [{y.min():.1f}, {y.max():.1f}]")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Features: ['age', 'sex', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6']
Dataset shape: (442, 10)
Target range: [25.0, 346.0]


## 2. Start MLflow & Train Model

In [4]:
mlflow.set_tracking_uri("http://localhost:5001")
mlflow.set_experiment("diabetes-prediction")

params = {
    "n_estimators": 100,
    "max_depth": 10,
    "random_state": 42,
}

with mlflow.start_run(run_name="random_forest_diabetes") as run:
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", RandomForestRegressor(**params)),
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    mlflow.log_params(params)
    mlflow.log_metrics({"mse": mse, "mae": mae, "r2": r2})

    mlflow.sklearn.log_model(
        pipeline,
        artifact_path="model",
        registered_model_name="diabetes",
    )

    run_id = run.info.run_id
    print(f"Run ID: {run_id}")
    print(f"MSE: {mse:.2f}")
    print(f"MAE: {mae:.2f}")
    print(f"R2:  {r2:.4f}")

2026/03/13 00:09:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/13 00:09:45 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'diabetes' already exists. Creating a new version of this model...
2026/03/13 00:09:47 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: diabetes, version 2


Run ID: 6bbaceb3a0e5424ca3075299d3a5813a
MSE: 2975.70
MAE: 44.43
R2:  0.4384
🏃 View run random_forest_diabetes at: http://localhost:5001/#/experiments/1/runs/6bbaceb3a0e5424ca3075299d3a5813a
🧪 View experiment at: http://localhost:5001/#/experiments/1


Created version '2' of model 'diabetes'.


## 3. Load Registered Model from MLflow

In [5]:
model_uri = "models:/diabetes/1"
loaded_model = mlflow.sklearn.load_model(model_uri)

sample = X_test[:3]
predictions = loaded_model.predict(sample)
print("Sample predictions from loaded model:")
for i, pred in enumerate(predictions):
    print(f"  Sample {i+1}: {pred:.2f} (actual: {y_test[i]:.2f})")

Sample predictions from loaded model:
  Sample 1: 144.96 (actual: 219.00)
  Sample 2: 172.95 (actual: 70.00)
  Sample 3: 154.18 (actual: 202.00)


## 4. Save Model for the Service

In [6]:
import pickle
from pathlib import Path

model_dir = Path("../mlapp/model")
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / "model.pkl"
with open(model_path, "wb") as f:
    pickle.dump(loaded_model, f)

print(f"Model saved to {model_path.resolve()}")

Model saved to /Users/litwein/GithubControl/HSE_DataOps/homework_24/mlapp/model/model.pkl
